In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression


In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score,recall_score, f1_score,confusion_matrix, classification_report,roc_auc_score, roc_curve)

In [ ]:
print("All libraries imported successfully!")

In [ ]:
hotel_df = pd.read_csv('hotel_bookings.csv')

In [ ]:
print(f"Total Records : {hotel_df.shape[0]:,}")

In [ ]:
print(f"Total Features: {hotel_df.shape[1]}")

In [ ]:
hotel_df.head()

In [ ]:
hotel_df.shape

# Data Preprocessing

In [ ]:
hotel_df.head()

In [ ]:
hotel_df.shape

### Duplicate Rows Check

In [ ]:
print("=" * 45)
print("DUPLICATE ROWS CHECK")
print("=" * 45)

# Total duplicates count
duplicates = hotel_df.duplicated().sum()
total      = len(hotel_df)

print(f"Total Rows        : {total:,}")
print(f"Duplicate Rows    : {duplicates:,}")
print(f"Duplicate %       : {duplicates/total*100:.2f}%")
print(f"Unique Rows       : {total - duplicates:,}")

if duplicates > 0:
    print(f"\n⚠️  {duplicates:,} duplicate rows found — need to remove!")
else:
    print("\n No duplicates found!")

## Duplicate Row Remove 

In [ ]:
before = len(hotel_df)
hotel_df = hotel_df.drop_duplicates()
after  = len(hotel_df)

print(f"Before : {before:,} rows")
print(f"After  : {after:,} rows")
print(f"Removed: {before - after:,} duplicate rows")

In [ ]:
hotel_df.duplicated()

In [ ]:
hotel_df.duplicated().sum()

### Null Values check

In [ ]:
hotel_df.isnull().sum()

In [ ]:
print("=" * 45)
print("HANDLING MISSING VALUES")
print("=" * 45)

# 1. Drop agent and company — too many nulls
hotel_df = hotel_df.drop(columns=['agent', 'company'])
print(" Dropped  : agent (12,193 nulls)")
print(" Dropped  : company (82,137 nulls)")

# 2. Fill children nulls with 0
hotel_df['children'] = hotel_df['children'].fillna(0)
print(" Filled   : children → 0")

# 3. Fill country nulls with mode
hotel_df['country'] = hotel_df['country'].fillna(
                      hotel_df['country'].mode()[0])
print(" Filled   : country → mode value")

# Verify — no nulls remaining
print()
print("=" * 45)
print("VERIFY — NULL VALUES AFTER FIX")
print("=" * 45)
remaining = hotel_df.isnull().sum().sum()
print(f"Remaining null values : {remaining}")

if remaining == 0:
    print(" All missing values handled!")
else:
    print(f" {remaining} nulls still remaining!")

In [ ]:
hotel_df.isnull().sum()

### Data Leakage Columns Drop

In [ ]:
print("=" * 45)
print("CORRELATION WITH TARGET (is_canceled)")
print("=" * 45)


numeric_df = hotel_df.select_dtypes(include='number')
correlation = numeric_df.corr()['is_canceled'].abs().sort_values(ascending=False)

print(correlation)

### Outlier Treatment

In [ ]:
print("=" * 45)
print("OUTLIER TREATMENT — adr COLUMN")
print("=" * 45)

# Before
print("BEFORE treatment:")
print(f"  Min adr : {hotel_df['adr'].min():.2f}")
print(f"  Max adr : {hotel_df['adr'].max():.2f}")
print(f"  Mean adr: {hotel_df['adr'].mean():.2f}")

# Visualise before
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.boxplot(hotel_df['adr'])
plt.title('ADR — Before Outlier Treatment')
plt.ylabel('Average Daily Rate ($)')

# Fix — cap at 99th percentile
adr_cap = hotel_df['adr'].quantile(0.99)
hotel_df['adr'] = hotel_df['adr'].clip(lower=0, upper=adr_cap)

# Visualise after
plt.subplot(1, 2, 2)
plt.boxplot(hotel_df['adr'])
plt.title('ADR — After Outlier Treatment')
plt.ylabel('Average Daily Rate ($)')

plt.tight_layout()
plt.savefig('outlier_treatment.png', dpi=150, bbox_inches='tight')
plt.show()

# After
print("\nAFTER treatment:")
print(f"  Min adr : {hotel_df['adr'].min():.2f}")
print(f"  Max adr : {hotel_df['adr'].max():.2f}")
print(f"  Mean adr: {hotel_df['adr'].mean():.2f}")
print(f"\n Outliers capped at 99th percentile: {adr_cap:.2f}")

### Label Encoding

In [ ]:
print("=" * 45)
print("LABEL ENCODING — TEXT TO NUMBERS")
print("=" * 45)

from sklearn.preprocessing import LabelEncoder

# Find all text (object) columns
cat_cols = hotel_df.select_dtypes(include='object').columns.tolist()

print(f"Text columns found : {len(cat_cols)}")
print(f"Columns            : {cat_cols}")
print()

# Encode each column
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    hotel_df[col] = le.fit_transform(hotel_df[col].astype(str))
    le_dict[col]  = le
    print(f" Encoded : {col}")

print()
print("=" * 45)
print("ENCODING VERIFICATION")
print("=" * 45)
print(f"Text columns remaining : "
      f"{hotel_df.select_dtypes(include='object').shape[1]}")
print(f"All numeric now        : "
      f"{hotel_df.select_dtypes(include='number').shape[1]} columns")
print()
print(" Label Encoding complete!")
print()

# Show sample encoded values
print("=" * 45)
print("SAMPLE — ENCODED VALUES")
print("=" * 45)
print(hotel_df[['hotel','meal','deposit_type','customer_type']].head())

### Define Features (X) and Target (y)

In [ ]:
print("=" * 45)
print("FEATURES (X) AND TARGET (y) SPLIT")
print("=" * 45)

X = hotel_df.drop('is_canceled', axis=1)
y = hotel_df['is_canceled']

print(f"Features (X) shape : {X.shape}")
print(f"Target   (y) shape : {y.shape}")
print()

# Feature columns show 
print("=" * 45)
print("FEATURE COLUMNS (X)")
print("=" * 45)
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2}. {col}")

print()
print("=" * 45)
print("TARGET COLUMN (y) — is_canceled")
print("=" * 45)
print(f"  Not Cancelled (0) : {(y==0).sum():,}")
print(f"  Cancelled     (1) : {(y==1).sum():,}")
print(f"  Cancel Rate       : {y.mean()*100:.1f}%")
print()
print(" X and y split complete!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Dataset Overview — Features & Target',
             fontsize=16, fontweight='bold')

# ── Chart 1: Target Distribution (Bar) ───────────────────
labels = ['Not Cancelled (0)', 'Cancelled (1)']
counts = [( y==0).sum(), (y==1).sum()]
colors = ['#4C72B0', '#DD8452']

bars = axes[0].bar(labels, counts, color=colors,
                   edgecolor='white', linewidth=0.8)
axes[0].set_title('Target Distribution',
                  fontweight='bold', fontsize=12)
axes[0].set_ylabel('Number of Bookings')
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 200,
                 f'{count:,}',
                 ha='center', fontweight='bold', fontsize=11)

# ── Chart 2: Target Distribution (Pie) ───────────────────
axes[1].pie(counts,
            labels=labels,
            autopct='%1.1f%%',
            colors=colors,
            startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Cancellation Rate',
                  fontweight='bold', fontsize=12)

# ── Chart 3: Feature Data Types ──────────────────────────
dtype_counts = X.dtypes.value_counts()
axes[2].bar(dtype_counts.index.astype(str),
            dtype_counts.values,
            color=['#55A868','#4C72B0'],
            edgecolor='white', linewidth=0.8)
axes[2].set_title('Feature Data Types',
                  fontweight='bold', fontsize=12)
axes[2].set_ylabel('Number of Columns')
for i, v in enumerate(dtype_counts.values):
    axes[2].text(i, v + 0.1, str(v),
                 ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('xy_split_overview.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Charts saved: xy_split_overview.png")

### Train / Test Split

In [ ]:
print("=" * 45)
print("TRAIN / TEST SPLIT")
print("=" * 45)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,      # 20% test
    random_state = 42,        # reproducibility
    stratify     = y          # class balance maintain
)

print(f"Total dataset    : {len(X):,} rows")
print()
print(f"Training set     : {X_train.shape[0]:,} rows "
      f"({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set         : {X_test.shape[0]:,} rows  "
      f"({X_test.shape[0]/len(X)*100:.0f}%)")
print()
print(f"Training features: {X_train.shape[1]} columns")
print(f"Test features    : {X_test.shape[1]} columns")
print()

# Class balance check
print("=" * 45)
print("CLASS BALANCE CHECK")
print("=" * 45)
print(f"Train — Not Cancelled: {(y_train==0).sum():,} "
      f"({(y_train==0).mean()*100:.1f}%)")
print(f"Train — Cancelled    : {(y_train==1).sum():,} "
      f"({(y_train==1).mean()*100:.1f}%)")
print()
print(f"Test  — Not Cancelled: {(y_test==0).sum():,}  "
      f"({(y_test==0).mean()*100:.1f}%)")
print(f"Test  — Cancelled    : {(y_test==1).sum():,}  "
      f"({(y_test==1).mean()*100:.1f}%)")
print()
print("Train/Test split complete!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Train / Test Split Overview',
             fontsize=14, fontweight='bold')

# Chart 1 — Split Size
split_labels = ['Train (80%)', 'Test (20%)']
split_sizes  = [X_train.shape[0], X_test.shape[0]]
split_colors = ['#4C72B0', '#DD8452']

axes[0].bar(split_labels, split_sizes,
            color=split_colors,
            edgecolor='white', linewidth=0.8)
axes[0].set_title('Dataset Split Size',
                  fontweight='bold')
axes[0].set_ylabel('Number of Rows')
for i, v in enumerate(split_sizes):
    axes[0].text(i, v + 300, f'{v:,}',
                 ha='center', fontweight='bold')

# Chart 2 — Class Balance
x      = np.arange(2)
width  = 0.35
train_counts = [(y_train==0).sum(), (y_train==1).sum()]
test_counts  = [(y_test==0).sum(),  (y_test==1).sum()]

axes[1].bar(x - width/2, train_counts,
            width, label='Train',
            color='#4C72B0', edgecolor='white')
axes[1].bar(x + width/2, test_counts,
            width, label='Test',
            color='#DD8452', edgecolor='white')
axes[1].set_title('Class Balance — Train vs Test',
                  fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Not Cancelled', 'Cancelled'])
axes[1].legend()

plt.tight_layout()
plt.savefig('train_test_split.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved: train_test_split.png")

### Feature Scaling

In [ ]:
print("=" * 45)
print("FEATURE SCALING — STANDARD SCALER")
print("=" * 45)
print()
print("Why scaling is needed:")
print("  lead_time range : 0 to 737")
print("  adr range       : 0 to 261")
print("  adults range    : 0 to 10")
print("  → Different scales confuse the model!")
print()

from sklearn.preprocessing import StandardScaler

# IMPORTANT: fit_transform on TRAIN only
# transform on TEST only — prevent data leakage!
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("After StandardScaler:")
print(f"  Training mean : {X_train_scaled.mean():.4f} (should be ~0)")
print(f"  Training std  : {X_train_scaled.std():.4f}  (should be ~1)")
print(f"  Test mean     : {X_test_scaled.mean():.4f}")
print(f"  Test std      : {X_test_scaled.std():.4f}")
print()
print(f"  X_train_scaled shape : {X_train_scaled.shape}")
print(f"  X_test_scaled shape  : {X_test_scaled.shape}")
print()
print("Feature Scaling complete!")